# RAG Pipeline Reliability Demo

This notebook builds a lightweight, local RAG-like reliability workflow.

- We simulate retrieval + token-confidence features per query.
- We train a classifier to predict answer correctness.
- We run `ReliabilityReport` to audit coverage, calibration, drift, and deficiency.

If the robustea deficiency backend is unavailable in your interpreter, the notebook still runs and reports `delta_hat = None`.

In [ ]:
import os
import sys
import numpy as np
from sklearn.linear_model import LogisticRegression

for _candidate in (os.getcwd(), os.path.abspath('..')):
    if os.path.isdir(os.path.join(_candidate, 'reliability')) and _candidate not in sys.path:
        sys.path.insert(0, _candidate)
        break

from pathlib import Path

from reliability import ReliabilityReport

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def simulate_rag_queries(n, shift=0.0, seed=0):
    rng = np.random.default_rng(seed)

    top1_retrieval = rng.beta(6 - shift, 3 + shift, size=n)
    mean_topk_retrieval = np.clip(top1_retrieval - rng.normal(0.08 + 0.05 * shift, 0.04, size=n), 0.0, 1.0)
    token_confidence = np.clip(rng.beta(7 - shift, 2 + shift, size=n), 0.01, 0.99)
    context_overlap = np.clip(rng.normal(0.65 - 0.10 * shift, 0.15, size=n), 0.0, 1.0)
    answer_len = rng.integers(20, 220, size=n).astype(float)

    logit = (
        -2.1
        + 3.4 * top1_retrieval
        + 2.8 * token_confidence
        + 1.2 * context_overlap
        - 0.0035 * answer_len
        - 0.9 * shift
    )
    p_correct = sigmoid(logit)
    y = (rng.uniform(0.0, 1.0, size=n) < p_correct).astype(int)

    X = np.column_stack([
        top1_retrieval,
        mean_topk_retrieval,
        token_confidence,
        context_overlap,
        answer_len,
    ])
    return X, y

# Train/calibration from baseline environment; test from shifted environment
X_train, y_train = simulate_rag_queries(3000, shift=0.0, seed=0)
X_cal, y_cal = simulate_rag_queries(1000, shift=0.0, seed=1)
X_test, y_test = simulate_rag_queries(1000, shift=0.9, seed=2)

rag_reliability_model = LogisticRegression(max_iter=2000, random_state=0)
rag_reliability_model.fit(X_train, y_train)

In [ ]:
report = ReliabilityReport(
    model=rag_reliability_model,
    X_cal=X_cal,
    y_cal=y_cal,
    X_test=X_test,
    y_test=y_test,
    alpha=0.10,
    task='classification',
    deficiency_n_bins=12,
    deficiency_n_bootstrap=10,
)

report.compute()
out_dir = Path('examples') if Path('reliability').is_dir() else Path('.')
report.to_html(str(out_dir / 'rag_reliability_report.html'))

metrics = report.to_dict()
{
    'status': metrics['summary']['status'],
    'empirical_coverage': metrics['summary']['empirical_coverage'],
    'ece_calibration': metrics['calibration']['ece'],
    'drift_detected': metrics['drift']['drift_detected'],
    'delta_hat': metrics['deficiency']['delta'],
}

## Adapting to a real RAG stack

Replace the synthetic features with real per-query logs from your pipeline:
- retrieval scores (top-1, top-k aggregate, reranker score)
- token-level confidence summaries (mean logprob, entropy, min token prob)
- metadata proxies (answer length, citation count, context overlap)

Then keep the same `ReliabilityReport` flow to monitor production shift and validity.